# Bengaluru Traffic Congestion: Exploratory Data Analysis & Feature Engineering

This notebook explores the **Bangalore Traffic Pulse / Bangalore City Traffic Dataset** to understand urban traffic dynamics across key corridors (Koramangala, Indiranagar, Whitefield, Electronic City, Silk Board, etc.).

### Objectives:
1. Inspect dataset structure, missing values, and data types.
2. Analyze traffic volume, average speed, travel time index (TTI), and road capacity utilization.
3. Understand congestion distribution across days of the week, weather conditions, and roadwork events.
4. Assess feature correlations and prevent target leakage for Random Forest modeling.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = Path.cwd().parent.parent
DATA_DIR = BASE_DIR / "data"
RAW_CSV = DATA_DIR / "raw" / "Banglore_traffic_Dataset.csv"
PROCESSED_CSV = DATA_DIR / "processed" / "cleaned_traffic.csv"

sys.path.insert(0, str(BASE_DIR / "ml"))
from preprocessing import load_raw_data, clean_data, TARGET_COLUMN, NUMERIC_FEATURES

## 1. Load Data
If the Kaggle CSV is present at `data/raw/Banglore_traffic_Dataset.csv`, it is loaded. Otherwise, the verified fallback demo generator provides a statistically aligned sample.

In [ ]:
df_raw, is_demo = load_raw_data()
print(f"Loaded dataset: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns. Demo mode: {is_demo}")
df_raw.head()

## 2. Data Cleaning & Feature Transformation

In [ ]:
df_clean = clean_data(df_raw, is_demo)
print(f"Cleaned records: {len(df_clean)}")
df_clean.describe()

## 3. Congestion Level Distribution

In [ ]:
cong_counts = df_clean[TARGET_COLUMN].value_counts()
print(cong_counts)

plt.figure(figsize=(7, 4))
sns.countplot(data=df_clean, x=TARGET_COLUMN, order=["LOW", "MEDIUM", "HIGH"], palette=["#16a34a", "#d97706", "#dc2626"])
plt.title("Distribution of Congestion Levels")
plt.xlabel("Congestion Level")
plt.ylabel("Count")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.show()

## 4. Speed vs Traffic Volume & Capacity Utilization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df_clean, x=TARGET_COLUMN, y="Average Speed (km/h)", order=["LOW", "MEDIUM", "HIGH"], ax=axes[0])
axes[0].set_title("Average Speed by Congestion Level")

sns.boxplot(data=df_clean, x=TARGET_COLUMN, y="Road Capacity Utilization (%)", order=["LOW", "MEDIUM", "HIGH"], ax=axes[1])
axes[1].set_title("Road Capacity Utilization (%) by Congestion Level")

plt.tight_layout()
plt.show()

## 5. Correlation Matrix & Leakage Check
Ensure no target leakage occurs into the model training pipeline.

In [ ]:
numeric_cols = [c for c in NUMERIC_FEATURES if c in df_clean.columns]
plt.figure(figsize=(9, 7))
sns.heatmap(df_clean[numeric_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f", cbar=True)
plt.title("Feature Correlation Matrix")
plt.show()